# Insider Threat Detection via UEBA — EDA & Evaluation

This notebook is the reproducible exploratory analysis and evaluation for the CERT r4.2 insider-threat UEBA project.

**Methodology (v4):** raw event logs → user-day behavioral features → 14-day personal baselines → feature z-scores → time-based train/test split → feature-set selection on a validation window inside the training period → Isolation Forest → hybrid rule and 7-day user risk → test-period evaluation per day and per insider.

Thirteen behavioral features are built from Logon, Device, File and Email activity. The model uses the subset chosen on the validation window (saved in `models/selected_features.json` by `python -m src.pipeline`). HTTP was tested separately and is excluded because it produced no measurable improvement while adding substantial I/O cost.

Run `python -m src.pipeline` before this notebook so the scored parquet and the feature selection file exist.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.config import DATA_RAW_DIR, DATA_PROCESSED_DIR

print("Project root:", PROJECT_ROOT)
print("Raw data directory:", DATA_RAW_DIR)

## 1. Dataset Overview

CERT r4.2 contains multiple behavioral event logs plus static employee-context tables. The final pipeline uses four event sources: **logon, device, file and email**. HTTP, LDAP and psychometric data are inspected for context but are not part of the production feature set.

In [ ]:
logon = pd.read_csv(DATA_RAW_DIR / "logon.csv", parse_dates=["date"])
device = pd.read_csv(DATA_RAW_DIR / "device.csv", parse_dates=["date"])
file_df = pd.read_csv(
    DATA_RAW_DIR / "file.csv",
    usecols=["id", "date", "user", "pc", "filename"],
    parse_dates=["date"],
)
email_df = pd.read_csv(
    DATA_RAW_DIR / "email.csv",
    usecols=["id", "date", "user", "pc", "to", "cc", "bcc", "from", "size", "attachments"],
    parse_dates=["date"],
)

print("Logon:", logon.shape)
print("Device:", device.shape)
print("File:", file_df.shape)
print("Email:", email_df.shape)

In [ ]:
for name, frame in {
    "Logon": logon,
    "Device": device,
    "File": file_df,
    "Email": email_df,
}.items():
    print(f"\n{name}")
    print("Date range:", frame["date"].min(), "to", frame["date"].max())
    print("Missing values:", int(frame.isna().sum().sum()))
    print("Duplicate IDs:", int(frame["id"].duplicated().sum()))

### Activity validation

The feature functions depend on the actual activity labels, so these are checked before aggregation.

In [ ]:
print("Logon activities:", sorted(logon["activity"].dropna().unique().tolist()))
print("Device activities:", sorted(device["activity"].dropna().unique().tolist()))

### Static context and large HTTP source

LDAP and psychometric data are retained as available context but are not wired into the current detector. HTTP is sampled here only for schema inspection; the full 14.5GB file is deliberately not scanned by the final EDA.

In [ ]:
ldap_dir = DATA_RAW_DIR / "LDAP"
ldap_files = sorted(ldap_dir.glob("*.csv"))
ldap = pd.concat([pd.read_csv(f) for f in ldap_files], ignore_index=True)
psychometric = pd.read_csv(DATA_RAW_DIR / "psychometric.csv")
http_sample = pd.read_csv(DATA_RAW_DIR / "http.csv", nrows=1000)

print("LDAP:", ldap.shape)
print("Psychometric:", psychometric.shape)
print("HTTP sample:", http_sample.shape)
print("HTTP columns:", http_sample.columns.tolist())

## 2. User-Day Feature Engineering

The detector converts event-level activity into one row per **user-day**. This reduces millions of raw events into behavioral observations that can be compared with each employee's own history.

In [ ]:
from src.features.user_day import logon_features, device_features, file_features, email_features

lf = logon_features(logon)
df_ = device_features(device)
ff = file_features(file_df)
ef = email_features(email_df)

print("Logon features:", lf.shape)
print("Device features:", df_.shape)
print("File features:", ff.shape)
print("Email features:", ef.shape)

display(lf.head())

Thirteen behavioral features are built:

- Logon: login count, after-hours logins, first login hour, distinct PCs used
- USB/device: USB connects, after-hours USB connects, device events
- File: file events, unique filenames, after-hours file events
- Email: email count, attachments, external-recipient ratio

A separate 0/1 flag, `first_usb_ever`, marks a user's first-ever USB day. It is not a model input; it drives the hybrid rule.

In [ ]:
from src.baselines.personal_baseline import FEATURE_COLS, merge_daily_features, add_personal_baseline

print("Final feature set:")
for feature in FEATURE_COLS:
    print("-", feature)

daily = merge_daily_features(lf, df_, ff, ef)
daily = add_personal_baseline(daily)

print("\nDaily table after personal baselines:", daily.shape)
display(daily[["user", "day"] + FEATURE_COLS].head())

### Raw feature distributions by outcome

Before introducing personal baselines, it's worth looking at a few raw (non-baselined) features split by whether the user-day is ground-truth malicious. This is a cheap sanity check on the project's central premise: raw magnitude alone should be a weak signal, which is exactly why comparing each user to their own history (rather than to the raw population distribution) is the right design choice.

In [ ]:
dist_cols = ["logon_count", "email_count", "usb_connect_count", "external_recipient_ratio"]
dist_source = pd.read_parquet(DATA_PROCESSED_DIR / "scored_daily.parquet")

fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, col in zip(axes.ravel(), dist_cols):
    for label, group in dist_source.groupby("is_malicious"):
        ax.hist(
            group[col].clip(upper=group[col].quantile(0.99)),
            bins=30,
            alpha=0.55,
            density=True,
            label="Malicious" if label == 1 else "Benign",
        )
    ax.set_title(col)
    ax.legend(fontsize=8)
fig.suptitle("Raw feature distributions: benign vs. malicious user-days")
fig.tight_layout()
plt.show()

Labels (`is_malicious`) come from the already-scored parquet so this check can be shown early. Raw distributions overlap heavily between benign and malicious days for most features, which is the expected result. It is the reason a *personal*, per-user rolling baseline (Section 3) is needed instead of comparing everyone to the same population-wide cutoff.

## 3. Personal Baselines

For each user and feature, the baseline is based on the user's previous 14 days. The implementation shifts the history before applying the rolling window, so the current day's behavior cannot leak into its own baseline.

This is the central UEBA idea: **an unusual day is judged primarily against that employee's own historical behavior rather than a population-wide average.**

In [ ]:
from src.baselines.deviation import add_deviation_features, statistical_anomaly_score

daily = add_deviation_features(daily)
daily["stat_anomaly_score"] = statistical_anomaly_score(daily)

zscore_cols = [f"{c}_zscore" for c in FEATURE_COLS]
print("Deviation features added:", len(zscore_cols))
display(daily[["user", "day", "stat_anomaly_score"] + zscore_cols].head())

## 4. Statistical Baseline: Max Absolute Z-Score

The statistical comparison uses the largest absolute feature z-score for each user-day. It is retained as a transparent baseline rather than the final detector.

In [ ]:
display(
    daily.sort_values("stat_anomaly_score", ascending=False)[
        ["user", "day", "stat_anomaly_score"] + FEATURE_COLS
    ].head(15)
)

A very large z-score is not automatically evidence of an attack. With a very small historical standard deviation, even a modest absolute change can produce a large z-score. This is one reason the project compares the transparent statistical baseline with a multivariate Isolation Forest.

## 5. Primary Model — Isolation Forest

Isolation Forest does not need malicious labels. It is fitted on the **training period only** (the first 60% of dates), using the feature set that won on the validation window. Cold-start rows (no baseline yet) are skipped rather than scored.

In [ ]:
from src.models.feature_selection import load_selected_columns
from src.models.isolation_forest_model import MODEL_COLS, fit_isolation_forest, score_isolation_forest, split_by_date
from src.utils.config import MODELS_DIR

selected_columns = load_selected_columns(MODELS_DIR / "selected_features.json") or MODEL_COLS
train_daily, test_daily, cutoff = split_by_date(daily)

model = fit_isolation_forest(train_daily, columns=selected_columns)
scored = score_isolation_forest(model, daily, columns=selected_columns)

print("Trained on days before:", cutoff)
print("Model inputs:", [c.replace("_zscore", "") for c in selected_columns])
print("Scored table:", scored.shape)
display(
    scored.sort_values("if_anomaly_score", ascending=False)[
        ["user", "day", "if_anomaly_score", "stat_anomaly_score"]
    ].head(15)
)

In [ ]:
import json

selection = json.loads((MODELS_DIR / "selected_features.json").read_text())
print("Selected on validation:", selection["selected"])
display(pd.DataFrame(selection["validation_results"]))

## 6. Ground-Truth Evaluation (test period only)

CERT answer files are converted into malicious user-day labels. Everything below uses the saved pipeline output. Thresholds always come from the **training** scores (99th percentile) and are applied to the **test** period, which the model never saw.

Scores compared:

- **Isolation Forest (daily)**: the model's score for that day
- **Hybrid**: the daily score plus the first-ever-USB-after-hours rule
- **7-day user risk (max)**: the highest daily score in the user's last 7 days
- **Combined**: alert if the daily score OR the 7-day risk is unusual (each converted to a training percentile), plus the rule, at the same 1% budget
- **Max |z| baseline**: the largest single z-score of the day

In [ ]:
from src.evaluation.evaluate import evaluate_scores, train_threshold
from src.evaluation.report import build_report, insiders_by_scenario, load_scored, scenario_recall

scored_full = load_scored()
test_full = scored_full[scored_full["split"] == "test"]
threshold = train_threshold(scored_full, "if_anomaly_score")

display(scored_full.groupby("split")["is_malicious"].agg(user_days="count", malicious_days="sum"))
print("Isolation Forest threshold (train 99th percentile):", round(threshold, 4))

In [ ]:
report_table = build_report(scored_full)
display(report_table)

**Headline result (v4, 26 Nov 2026 run):** on the test period, the hybrid model catches **21 of 34 insiders (62%)** with at least one alert on a malicious day, at about 8 alerts a day for the whole organisation. Day-level precision is 0.030 (about 9× random guessing, since only 0.34% of test days are malicious) and PR-AUC is 0.024. Most alerts are still false positives, so this is an analyst-assistive detector, not an automated one. Re-run the cell above after `python -m src.pipeline` to confirm the exact figures.

### Scenario-level detection

Overall metrics hide differences between attack scenarios. Day-level recall can look low for long-running scenarios, so detection is also counted per insider.

In [ ]:
display(scenario_recall(scored_full))
display(insiders_by_scenario(scored_full))

### Why the first-ever-USB rule did or did not catch r4.2-1 insiders

The rule fires on a user's first-ever USB day when there is also after-hours activity. The table explains, insider by insider, why it fired or missed.

In [ ]:
from src.evaluation.diagnosis import rule_diagnosis

diagnosis = rule_diagnosis(scored_full)
display(diagnosis)
display(diagnosis["why"].value_counts())

### Evaluation visuals

How imbalanced the test classes are, how the scores compare across the full precision/recall trade-off, how many insiders are caught at different alert budgets, and the confusion matrix at the training-derived threshold.

In [ ]:
from src.evaluation.plots import (
    plot_class_balance,
    plot_precision_recall_comparison,
    plot_confusion_matrix,
)

plot_class_balance(test_full)
plt.show()

In [ ]:
plot_precision_recall_comparison(
    test_full,
    {
        "Isolation Forest (daily)": "if_anomaly_score",
        "Hybrid": "hybrid_score",
        "7-day user risk": "user_risk_7d",
        "Statistical baseline": "stat_anomaly_score",
    },
)
plt.show()

The alert-budget table converts each threshold (a training percentile) into alerts per day on the test period, so a security team can choose a threshold from its review capacity.

In [ ]:
from src.evaluation.analysis import alert_budget, plot_alert_budget
from src.evaluation.report import SCORES

budgets = {name: alert_budget(scored_full, col) for name, col in SCORES.items()}
display(budgets["Isolation Forest (daily)"])
plot_alert_budget(budgets)
plt.show()

In [ ]:
plot_confusion_matrix(test_full, "if_anomaly_score", threshold)
plt.show()

## 7. Cold-Start Check

Users with fewer than three earlier days have no personal baseline. Their z-scores stay empty and those days are not scored, instead of being treated as perfectly normal. This check shows how many malicious days that affects.

In [ ]:
from src.evaluation.evaluate import attach_labels

daily_labeled = attach_labels(daily)
cold_start = daily_labeled[daily_labeled["logon_count_baseline_mean"].isna()]

print(cold_start["user"].nunique(), "users affected")
print(len(cold_start), "user-days with no score")
print(int(cold_start["is_malicious"].sum()), "of those days are ground-truth malicious")

## 8. Case Study — User Timeline

A timeline makes the detector easier to interpret than a metric alone: the anomaly score can be viewed alongside the malicious ground-truth periods.

In [ ]:
from src.evaluation.timeline import plot_user_timeline

case_user = (
    test_full[test_full["is_malicious"] == 1]["user"]
    .value_counts()
    .index[0]
)

print("Case-study user:", case_user)
fig = plot_user_timeline(scored_full, case_user, "if_anomaly_score", threshold=threshold)
plt.show()

### Population-level temporal pattern

The case study above looks at one user in depth. It's also worth checking whether alerts cluster in time in a way that would matter operationally — for example, a day-of-week effect, which would change how an analyst staffs review capacity.

In [ ]:
alerts = test_full[test_full["if_anomaly_score"] >= threshold].copy()
alerts["day"] = pd.to_datetime(alerts["day"])
alerts["day_of_week"] = alerts["day"].dt.day_name()

order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
counts = alerts["day_of_week"].value_counts().reindex(order).fillna(0)

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(counts.index, counts.values, color="#257de0")
ax.set(title="Test-period alerts by day of week", ylabel="Alert count")
ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.show()

## 9. Error Analysis

On the test period, at the training-derived threshold, the project distinguishes true positives, false positives, false negatives and true negatives. For false positives and false negatives, the largest absolute z-scores show which behavioral dimensions contributed most. The `top_reasons` column gives the same explanation in plain English for every scored day.

In [ ]:
from src.evaluation.error_analysis import categorize_errors, summarize_errors

labeled = categorize_errors(test_full, "if_anomaly_score", threshold)
print(labeled["error_type"].value_counts())

errors = summarize_errors(labeled, zscore_cols)

print("False negatives:")
display(errors[errors["error_type"] == "false_negative"].head(10))

print("False positives:")
display(errors[errors["error_type"] == "false_positive"].head(10))

In [ ]:
display(
    test_full[test_full["if_anomaly_score"] >= threshold]
    .nlargest(10, "if_anomaly_score")[["user", "day", "if_anomaly_score", "is_malicious", "top_reasons"]]
)

## 10. What the Experiments Established

### Final model decision
- **Isolation Forest on the validation-selected feature set** is the primary anomaly score, fitted on the training period only.
- A **hybrid rule** (first-ever USB use with after-hours activity) is added on top. It fires on very few days, so it barely uses the alert budget.
- The **7-day user risk (max)** and the **combined queue** are evaluated alongside. The combined queue uses training percentiles instead of hand-picked weights and keeps the same 1% budget.
- The max absolute z-score is kept as a transparent statistical comparison.

### What changed along the way
- v2 fixed two silent z-score bugs (cold-start rows scored as normal, zero-variance users hiding first-time behaviour) and moved to a forward-looking time split. Every metric improved.
- Counting per insider showed that r4.2-2 was caught well (14 of 16), and the real gap was r4.2-1.
- v3 added r4.2-1 features. They helped r4.2-1 but crowded out r4.2-2 alerts (12 of 34 insiders overall). A test-period ablation showed that fewer inputs did better.
- v4 chose the feature set on a validation window instead of the test set and turned the first-ever-USB flag into a rule, reaching 21 of 34 insiders.

### Investigated alternatives
- A PyTorch autoencoder did not outperform Isolation Forest.
- An anchor-baseline experiment did not materially improve r4.2-2.
- HTTP features left r4.2-2 effectively unchanged, so HTTP stays out of the pipeline.

### Main limitation
Day-level recall is still low, and most alerts are false positives. The daily score and the 7-day risk catch different scenarios, which the combined queue tries to exploit. r4.2-1 remains the hardest scenario for the daily score; the rule diagnosis above explains why.

## 11. Final Takeaway

This project demonstrates a practical UEBA workflow built around **behavioral baselines rather than signatures**. The final system converts raw CERT events into user-day behavior, measures deviation from each user's own history, applies an unsupervised Isolation Forest, and evaluates the resulting alerts against CERT ground truth.

The most important engineering result is not a single headline metric. It is the combination of **leakage-safe baselines, reproducible evaluation, model comparison, scenario-level analysis, and explicit documentation of failure modes**.